# SOLUTIONS · Laptop lab 5: Catching fraud
*When accuracy lies, and the threshold is a choice*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 26, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab05_solutions.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you learned to measure a classifier properly (Lecture 9) and what to do when the class you look for is rare (Lecture 10). Today you use both on real card payments.

**The story.** A bank wants to stop card fraud. A model flags payments, and an analyst checks every flagged payment and calls the cardholder: a few minutes of work, so a false alarm is cheap. But every fraud the model misses is money stolen from a customer. Only about **1 payment in 100** is fraud. As always, some payments stay locked away: the **test set**, 20,000 payments.

**Your mission.** Build one pipeline, and choose its threshold, so that it catches **at least 85% of the frauds** (recall ≥ 0.85) with **as few false alarms as possible** (precision as high as possible).

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. A check below the cell shows an orange *Not yet* box until you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

Today a model does not only say *fraud or not*; it says *how likely*. You choose where to draw the line:

```python
proba = model.predict_proba(X)[:, 1]   # the probability of fraud, for every payment
flag = proba >= 0.3                    # your threshold: flag a payment from 30% on
recall_score(y, flag)                  # the share of frauds that are flagged
precision_score(y, flag)               # the share of flagged payments that are fraud
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1–3. Warm-up: accuracy, the threshold, the probabilities | together on the screen, 15 |
| 4. Your mission | 45 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, and downloads the tools and the data of this lab. You should see a green box with a ✓.

In [ ]:
try:
    import imblearn                         # imbalanced-learn, for Lecture 10
except ImportError:
    !pip install -q imbalanced-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (recall_score, precision_score, average_precision_score,
                             precision_recall_curve, ConfusionMatrixDisplay, classification_report)
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier

# The tools and the data of this lab are in the course lab repository.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab05_tools.py", "lab05_tools.py")
from lab05_tools import *

X, y = load_training_data()    # your 30,000 payments: features X, and y = 1 for fraud
setup_ok()

**Run.** Your data: card payments of European cardholders over two days in September 2013. `Time` counts the seconds since the first payment, and `Amount` is the amount paid. To protect the cardholders, the bank did not publish the original features (shop, country, card type, ...), only 28 anonymous mixtures of them: `V1` to `V28`. There are no gaps and no text columns, so you can start right away.

In [ ]:
print(X.shape, "  frauds:", y.sum(), f"({y.mean():.1%})")
X.head()

## Warm-up: the pieces, one by one (Tasks 1 to 3)

We go through Tasks 1 to 3 together on the screen: just run the cells and follow along. You will reuse them in your mission.

## Task 1. Accuracy lies

**Run.** A model that never flags anything, against a logistic regression. Which one is better? (Lecture 9, "Is 0.87 good?")

In [ ]:
never = DummyClassifier()                                               # always "not fraud"
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

for name, model in [("never fraud", never), ("logistic regression", logreg)]:
    print(f"{name:20} mean validation accuracy {cross_val_score(model, X, y, cv=5).mean():.3f}")

Doing nothing is 99% accurate: 99 in 100 payments are honest anyway. The logistic regression looks only a little better. Accuracy hides what the bank cares about. Let us count the mistakes instead.

**Run.** `cross_val_predict` gives, for every payment, the prediction of the model that did *not* see it during training: an honest prediction for all 30,000 payments at once. The confusion matrix counts the four outcomes (Lecture 9, "The confusion matrix").

In [ ]:
predicted = cross_val_predict(logreg, X, y, cv=5)
ConfusionMatrixDisplay.from_predictions(y, predicted, display_labels=["honest", "fraud"])
plt.show()
print(f"recall    {recall_score(y, predicted):.2f}   (the share of frauds that are caught)")
print(f"precision {precision_score(y, predicted):.2f}   (the share of flagged payments that are fraud)")

When the model flags a payment, it is almost always right. But it lets about one fraud in five through: below the goal of 85%.

**Run.** Precision and recall are always measured for one class, the *positive* one. `classification_report` shows them for both classes:

In [ ]:
print(classification_report(y, predicted, target_names=["honest", "fraud"]))

The row of the honest payments looks perfect: almost every honest payment is recognized. That is easy when 99 in 100 payments are honest. The row of the frauds tells the real story. Call the other class positive and the false alarms and the misses swap places, so always read the row of the class you look for: here, fraud (Lecture 9, "Precision and recall").

## Task 2. The threshold is a choice

**Run.** `predict` flags a payment when the probability of fraud is at least 0.5. That is a choice, not a law (Lecture 9, "The threshold is a choice"). Lower the threshold, and watch recall and precision:

In [ ]:
proba = cross_val_predict(logreg, X, y, cv=5, method="predict_proba")[:, 1]   # honest probabilities

for threshold in [0.5, 0.1, 0.05, 0.02, 0.01]:
    flag = proba >= threshold
    print(f"threshold {threshold:.2f}: flags {flag.sum():5} payments,  "
          f"recall {recall_score(y, flag):.2f},  precision {precision_score(y, flag):.2f}")

A lower threshold catches more frauds, and raises more false alarms: slowly at first, then fast. **Run.** The precision-recall curve shows every threshold at once; the dot is the threshold 0.5. Average precision summarizes the whole curve in one number: 1 is perfect, and a model that guesses gets the share of frauds, 0.01 (Lecture 9, "The precision-recall curve").

In [ ]:
precision, recall, thresholds = precision_recall_curve(y, proba)
flag = proba >= 0.5

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(recall, precision, label=f"logistic regression (average precision {average_precision_score(y, proba):.2f})")
ax.plot(recall_score(y, flag), precision_score(y, flag), "o", color="black", label="threshold 0.5")
ax.axvline(0.85, color="grey", linestyle="--", label="the goal: recall 0.85")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend(loc="lower left")
plt.show()

To reach the goal, walk along the curve to the right until recall is 0.85, and read off the precision there. Just after the goal, the curve falls off a cliff: the last frauds look like honest payments. That is Step 2 of your mission.

## Task 3. Where the line falls

**Run.** Each dot is one payment, placed at its probability of fraud from Task 2: honest payments in the bottom row, frauds in the top row. A payment is flagged when it lies right of the line. The dashed lines are the thresholds 0.5 and 0.05 (Lecture 9, "The threshold is a choice").

In [ ]:
rng = np.random.default_rng(0)                       # a little up-and-down jitter, so that dots do not hide each other
fig, ax = plt.subplots(figsize=(8, 2.6))
for label, color in [(0, "#25567B"), (1, "#C0002A")]:
    p = proba[y == label]
    ax.scatter(p, label + rng.uniform(-0.3, 0.3, len(p)), s=5, alpha=0.3, color=color)
for threshold in [0.5, 0.05]:
    ax.axvline(threshold, color="grey", linestyle="--")
ax.set_yticks([0, 1], ["honest", "fraud"])
ax.set_xlabel("probability of fraud")
plt.show()

for threshold in [0.5, 0.05]:
    flag = proba >= threshold
    print(f"line at {threshold}: {flag[y == 1].sum()} of the {y.sum()} frauds are right of it, "
          f"and {flag[y == 0].sum()} honest payments")

Most honest payments pile up at 0, and most frauds sit near 1: those are easy. The hard cases lie in between. Moving the line from 0.5 to 0.05 takes in the frauds just left of 0.5, but also the honest payments there. The model is the same; only the line moved. And look at the frauds at the far left, among the honest payments: no line catches them without bothering thousands of honest customers. That is why the last frauds are so expensive to catch. Every threshold is a trade between the frauds you catch and the honest customers you bother.

## Task 4. Your mission

Now it is your turn. Build **one pipeline** and choose **one threshold** so that, in cross-validation, it catches **at least 85% of the frauds** with the **highest precision** you can get.

You do this in four short steps. Step 1 is ready to run. Steps 2 to 4 are split into small cells. Each cell says what to write, with a pointer to the **toolbox** right below; after it, a **Check** says what you should see and what it means. After every step, write down two numbers: the threshold that reaches recall 0.85, and the precision there.

**The rule:** you may do anything with `X` and `y`. The test set stays locked.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

### 1. Probabilities instead of labels

`predict` says 0 or 1. `predict_proba` gives two numbers for every payment: the probability that it is honest, and the probability that it is fraud. Column 1 is the probability of fraud.

```python
my_pipe.fit(X, y)
my_pipe.predict_proba(X)[:3]             # three rows of [probability honest, probability fraud]
```

A model is too sure about the payments it learned from. For honest probabilities, use `cross_val_predict`: every payment gets its probability from the model that did *not* see it during training.

```python
proba = cross_val_predict(my_pipe, X, y, cv=5, method="predict_proba")[:, 1]   # one number per payment
print(f"smallest {proba.min():.3f}, largest {proba.max():.3f}")              # all between 0 and 1
```

### 2. Recall and precision at a threshold you choose

Flag every payment whose probability is at least the threshold, and count:

```python
flag = proba >= 0.1             # True for every payment you flag
print(flag.sum())               # how many payments you flag
print(recall_score(y, flag))    # the share of the frauds that you flag
print(precision_score(y, flag)) # the share of the flagged payments that are fraud
print((flag & (y == 0)).sum())  # how many flagged payments are honest: the false alarms
```

Any threshold works the same way, also one you found with part 3: `flag = proba >= thresholds[reach]`.

### 3. The threshold for the goal

Instead of trying thresholds one by one, `precision_recall_curve` tries all of them at once. It gives three arrays: with threshold `thresholds[i]`, you get precision `precision[i]` and recall `recall[i]`. (`precision` and `recall` have one extra value at the end, without a threshold; `[:-1]` leaves it out.)

The thresholds go up, so recall goes down. Take the **last** position where recall is still at least 0.85: the highest threshold that reaches the goal, so the fewest false alarms.

```python
precision, recall, thresholds = precision_recall_curve(y, proba)
print(len(thresholds), "thresholds")              # how many thresholds it tried
reach = np.where(recall[:-1] >= 0.85)[0][-1]      # the last position with recall >= 0.85
print(f"threshold {thresholds[reach]:.3f}: recall {recall[reach]:.2f}, precision {precision[reach]:.2f}")
```

You will do this for several models, so a cell before Step 3 puts these lines in a small function for you:

```python
def at_goal(p, goal=0.85):
    # the highest threshold with recall >= goal, and the precision there
    precision, recall, thresholds = precision_recall_curve(y, p)
    reach = np.where(recall[:-1] >= goal)[0][-1]
    return thresholds[reach], precision[reach]

threshold, prec = at_goal(proba)                  # any probabilities: proba, proba_remedy, proba_forest
print(f"for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")
```

### 4. Class weights

One argument of the model. With `class_weight="balanced"`, every fraud counts as much as about a hundred honest payments during training (Lecture 10, "Remedy 2: class weights").

```python
my_remedy = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
my_remedy                                                                               # shows the pipeline
proba_remedy = cross_val_predict(my_remedy, X, y, cv=5, method="predict_proba")[:, 1]   # as in part 1
print(f"smallest {proba_remedy.min():.3f}, largest {proba_remedy.max():.3f}")
flag = proba_remedy >= 0.5                                                              # as in part 2
print(f"at 0.5: recall {recall_score(y, flag):.2f}, precision {precision_score(y, flag):.2f}")
```

### 5. Resampling, inside the pipeline

Change the training data instead of the model (Lecture 10, "Three ways to resample"). SMOTE makes up new frauds between neighboring frauds; undersampling drops honest payments until there are as many as frauds. Use the pipeline of **imbalanced-learn**, `make_imb_pipeline`: it resamples the training folds only, never the payments the model is tested on (Lecture 10, "Resample the training data only").

```python
# SMOTE:
my_remedy = make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression(max_iter=1000))
# or undersampling:
my_remedy = make_imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=0), LogisticRegression(max_iter=1000))
```

Use one of the two. Then get `proba_remedy` and use `at_goal` exactly as for class weights:

```python
proba_remedy = cross_val_predict(my_remedy, X, y, cv=5, method="predict_proba")[:, 1]
threshold, prec = at_goal(proba_remedy)
print(f"for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")
```

SMOTE measures distances, so the scaler comes first.

### 6. Forests

A forest needs no scaler, so the model alone is enough: use it wherever you used `my_pipe`. A balanced random forest gives every tree its own balanced sample (Lecture 10, "Undersampling inside a forest"). `n_jobs=-1` uses all processors; each forest still takes up to a minute in cross-validation.

```python
my_forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
balanced = BalancedRandomForestClassifier(n_estimators=100, sampling_strategy="all", replacement=True,
                                          bootstrap=False, random_state=0, n_jobs=-1)
proba_forest = cross_val_predict(my_forest, X, y, cv=5, method="predict_proba")[:, 1]
forest_threshold, forest_precision = at_goal(proba_forest)     # part 3
print(f"forest, for recall 0.85: threshold {forest_threshold:.3f}, precision {forest_precision:.2f}")

proba_balanced = cross_val_predict(balanced, X, y, cv=5, method="predict_proba")[:, 1]   # the balanced forest
threshold, prec = at_goal(proba_balanced)
```

### 7. Comparing models without a threshold

Average precision scores the whole precision-recall curve in one number: 1 is perfect, and guessing gets the share of frauds, about 0.01 (Lecture 9, "Comparing models without choosing a threshold"). Compute it from the honest probabilities you already have:

```python
print(f"average precision: forest {average_precision_score(y, proba_forest):.2f}, Step 1 {average_precision_score(y, proba):.2f}")
```

Two models can have the same average precision and still differ near the goal, so look at both numbers.

</details>

### Step 1. The starting pipeline

**Run.** The logistic regression of Task 1, at the usual threshold 0.5. Does it reach the goal?

In [ ]:
# Step 1. Run this cell. Nothing to change.
my_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
proba = cross_val_predict(my_pipe, X, y, cv=5, method="predict_proba")[:, 1]
flag = proba >= 0.5
print(f"Step 1, threshold 0.5: recall {recall_score(y, flag):.2f}, precision {precision_score(y, flag):.2f}")

*Stuck?* Cheat sheet, "Precision, recall, F1" · Lecture 9, "Precision and recall".

### Step 2. Choose the threshold

Keep the model of Step 1, and lower its threshold until it catches 85% of the frauds. You do it in three small cells.

**2a. The curve.** **Your turn.** Compute the precision-recall curve of the model of Step 1 from its probabilities `proba` (Toolbox, part 3). Print how many thresholds it tried.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y, proba)
print(len(thresholds), "thresholds")

**Check.** You get three arrays, and thousands of thresholds. `precision[i]` and `recall[i]` are what you get when you flag from `thresholds[i]` on.

**2b. The threshold for the goal.** **Your turn.** Find the highest threshold whose recall is still at least 0.85 (Toolbox, part 3), and print the threshold, its recall and its precision.

In [ ]:
reach = np.where(recall[:-1] >= 0.85)[0][-1]
print(f"Step 2, threshold {thresholds[reach]:.3f}: recall {recall[reach]:.2f}, precision {precision[reach]:.2f}")

**Check.** The threshold is far below 0.5 (around 0.05), and the recall is at least 0.85. The precision fell from 0.95 in Step 1 to about 0.70: that is the price of the goal.

**2c. Count the false alarms.** **Your turn.** Flag the payments with this threshold (Toolbox, part 2). How many payments do you flag, and how many of them are honest?

In [ ]:
flag = proba >= thresholds[reach]
print("flagged:", flag.sum(), " of which honest:", (flag & (y == 0)).sum())
# About 360 flagged, about 110 of them honest: an analyst calls about 110 honest customers to
# catch about 250 frauds.

**Check.** About one flagged payment in three is honest. Every one of them is a call to a customer who did nothing wrong. Can a better model catch the same frauds with fewer of those calls? That is what Steps 3 and 4 try. Write the threshold and the precision of Step 2 in your notes.

*Stuck?* Toolbox, parts 2 and 3 · Cheat sheet, "Probabilities and the threshold" · Lecture 10, "Remedy 1: lower the threshold".

**Run.** The lines of 2a and 2b as a small function, so that you can reuse them: `at_goal(p)` gives the threshold for recall 0.85 and the precision there.

In [ ]:
def at_goal(p, goal=0.85):
    # the highest threshold with recall >= goal, and the precision there (as in 2a and 2b)
    precision, recall, thresholds = precision_recall_curve(y, p)
    reach = np.where(recall[:-1] >= goal)[0][-1]
    return thresholds[reach], precision[reach]

threshold, prec = at_goal(proba)
print(f"Step 1 model, for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")

### Step 3. A remedy of Lecture 10

**3a. The pipeline.** **Your turn.** Make the pipeline of Step 1 again, but with class weights (Toolbox, part 4). Call it `my_remedy`.

In [ ]:
my_remedy = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
my_remedy

**Check.** Display `my_remedy`: you should see `class_weight='balanced'` in the logistic regression.

**3b. Honest probabilities.** **Your turn.** Get the probabilities of fraud of `my_remedy` with `cross_val_predict`, as in Step 1. Call them `proba_remedy`, and print the largest and the smallest.

In [ ]:
proba_remedy = cross_val_predict(my_remedy, X, y, cv=5, method="predict_proba")[:, 1]
print(f"smallest {proba_remedy.min():.3f}, largest {proba_remedy.max():.3f}")

**Check.** One number between 0 and 1 for each of the 30,000 payments.

**3c. At the threshold 0.5.** **Your turn.** Flag with 0.5, and print the recall and the precision.

In [ ]:
flag = proba_remedy >= 0.5
print(f"Step 3, at 0.5: recall {recall_score(y, flag):.2f}, precision {precision_score(y, flag):.2f}")

**Check.** The recall is now above the goal, already at 0.5. But look at the precision: only about one flagged payment in four is fraud. The weights pushed every probability of fraud up, so 0.5 flags far more payments. Is that better than Step 2?

**3d. At the goal.** **Your turn.** Use `at_goal` on `proba_remedy`, and compare the precision with Step 2.

In [ ]:
threshold, prec = at_goal(proba_remedy)
print(f"Step 3, for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")
# About the same precision as Step 2 (0.67 against 0.70), at a much higher threshold.

**Check.** At recall 0.85 the precision is about the same as in Step 2, or a little lower. The class weights did not make the model better: they only moved its threshold, along the same curve.

**3e. Try another remedy.** **Your turn.** Do 3a to 3d again with SMOTE (Toolbox, part 5). Does it do better at recall 0.85?

In [ ]:
my_remedy = make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression(max_iter=1000))
proba_remedy = cross_val_predict(my_remedy, X, y, cv=5, method="predict_proba")[:, 1]
threshold, prec = at_goal(proba_remedy)
print(f"Step 3, SMOTE, for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")
# The same as class weights (0.67). Undersampling is worse (0.54): it throws away most honest payments.

**Check.** About the same as class weights. The remedies choose another point on the same curve; choosing the threshold of the plain model in Step 2 was just as good.

*Stuck?* Toolbox, parts 4 and 5 · Cheat sheet, "A rare class" · Lecture 10, "Remedy 2: class weights" and "SMOTE: synthetic examples".

### Step 4. A forest

In Lecture 10, a better model mattered more than any remedy. Does it here?

**4a. The forest and its probabilities.** **Your turn.** Make a random forest with 100 trees, `random_state=0` and `n_jobs=-1` (Toolbox, part 6), call it `my_forest`, and get its probabilities `proba_forest` with `cross_val_predict`. This takes about a minute.

In [ ]:
my_forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
proba_forest = cross_val_predict(my_forest, X, y, cv=5, method="predict_proba")[:, 1]

**4b. The whole curve.** **Your turn.** Print the average precision of the forest, and of the model of Step 1 (Toolbox, part 7).

In [ ]:
print(f"average precision: forest {average_precision_score(y, proba_forest):.2f}, Step 1 {average_precision_score(y, proba):.2f}")

**Check.** About the same: over all thresholds together, the two models are equally good. But the bank uses only one threshold.

**4c. At the goal.** **Your turn.** Use `at_goal` on `proba_forest`. Call the results `forest_threshold` and `forest_precision`, and compare the precision with Step 2.

In [ ]:
forest_threshold, forest_precision = at_goal(proba_forest)
print(f"Step 4, for recall 0.85: threshold {forest_threshold:.3f}, precision {forest_precision:.2f}")
# Precision 0.77 against 0.70 in Step 2: near the goal, the forest raises fewer false alarms.

**Check.** At recall 0.85, the forest has a higher precision than the model of Step 2: fewer calls to honest customers for the same frauds. Average precision compares whole curves; the precision at the goal compares the part of the curve that the bank uses.

**4d. Optional: a balanced random forest.** Try the balanced random forest of Toolbox, part 6, with 4a and 4c.

In [ ]:
balanced = BalancedRandomForestClassifier(n_estimators=100, sampling_strategy="all", replacement=True,
                                          bootstrap=False, random_state=0, n_jobs=-1)
proba_balanced = cross_val_predict(balanced, X, y, cv=5, method="predict_proba")[:, 1]
threshold, prec = at_goal(proba_balanced)
print(f"Step 4, balanced forest, for recall 0.85: threshold {threshold:.3f}, precision {prec:.2f}")
# Precision 0.73: better than the logistic regression, a little below the plain forest.
# Our choice: the random forest, at its threshold for recall 0.85, about 0.12.

**Check.** Which of your pipelines has the highest precision at recall 0.85? That is the one you submit.

*Stuck?* Toolbox, parts 6 and 7 · Cheat sheet, "Curves" and "A rare class" · Lecture 10, "Undersampling inside a forest" and "The forests as curves".

### Your notes

**Your turn.** Double-click the cell below and write down, for each step, the threshold for recall 0.85 and the precision there. Then choose your pipeline and its threshold.

**Answer.** Step 1, threshold 0.5: recall 0.78, precision 0.95, below the goal. Step 2, the same model at threshold 0.047: recall 0.85, precision 0.70. Step 3, class weights and SMOTE catch more than 90% of the frauds at 0.5, but with a precision of only 0.25; at recall 0.85 their precision is 0.67, no better than Step 2, and undersampling gets 0.54. Step 4, all models have an average precision of 0.86 to 0.87, but at recall 0.85 the random forest reaches a precision of 0.77 and the balanced forest 0.73. Our choice: the random forest, with threshold 0.12.

### Your submission

**Your turn.** Fill in your pipeline, its threshold, and the recall and precision you expect on the locked payments (your cross-validated numbers are good estimates). The last line fits your pipeline once more on **all** 30,000 training payments.

In [ ]:
my_pipeline = my_forest                            # Step 4: random forest
my_threshold = round(float(forest_threshold), 3)   # the threshold for recall 0.85
my_recall = 0.85
my_precision = round(float(forest_precision), 2)

my_pipeline.fit(X, y)  # learn from all 30,000 training payments

In [ ]:
# Check
check_submission()

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare with your neighbor: the same model? The same threshold?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it flags the 20,000 payments that no model has seen with your threshold, and counts.

In [ ]:
X_test, y_test = unlock_test_set("fraudfind")    # the password on the screen

In [ ]:
test_recall, test_precision = report(my_pipeline, my_threshold, my_recall, my_precision, X_test, y_test)

Tell the tutor your pipeline, your threshold, and your recall and precision on the test set. We collect them on the screen: who met the goal, and with how many false alarms?

## Task 6. What happened?

**Run.** The pieces of today on the test set: each pipeline at the threshold 0.5, and its average precision. This takes about a minute.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
candidates = {
    "never fraud": never,
    "logistic regression": logreg,
    "  + class weights": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "  + SMOTE": make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression(max_iter=1000)),
    "random forest": RandomForestClassifier(100, random_state=0, n_jobs=-1),
    "balanced random forest": BalancedRandomForestClassifier(
        100, sampling_strategy="all", replacement=True, bootstrap=False, random_state=0, n_jobs=-1),
}
rows = []
for name, m in candidates.items():
    m.fit(X, y)
    p = m.predict_proba(X_test)[:, 1]
    rows.append({"pipeline": name,
                 "accuracy": ((p >= 0.5) == y_test).mean(),
                 "recall at 0.5": recall_score(y_test, p >= 0.5),
                 "precision at 0.5": precision_score(y_test, p >= 0.5, zero_division=0),
                 "average precision": average_precision_score(y_test, p)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) Which pipelines have an accuracy above 0.99, and would the bank want each of them? (Lecture 9, "Which metric?")<br>
(b) With class weights or SMOTE, recall at 0.5 goes up from about 0.84 to about 0.91, and precision falls from about 0.95 to about 0.25. Average precision hardly moves. What does that tell you? (Lecture 10, "The forests as curves")<br>
(c) You chose your threshold so that recall was 0.85 in cross-validation. Why not aim at 0.95, to be safe? Look at the curve of Task 2.<br>
(d) A shop wants to block a payment at the till, with no analyst in between, and a wrong block loses a customer. Which metric matters now, and would you move the threshold up or down?<br>
(e) Suppose a check costs the bank €5 and a missed fraud €100. What did your pipeline cost on the test set, and what would "never fraud" have cost?

**Answer.** (a) "Never fraud" (0.990), the logistic regression and the random forest (both 0.998). The bank does not want "never fraud": it catches no fraud at all. The other two catch about 85% of the frauds at 0.5, close to the goal. Class weights and SMOTE have the lowest accuracy, 0.97, yet catch the most frauds at 0.5. Accuracy cannot tell these apart, because 99 in 100 payments are honest anyway. (b) The remedies move the point along the curve, trading precision for recall, but they do not make the model better at ranking the payments: the curves, and so the average precision (0.88 to 0.89), stay about the same. Lowering the threshold of the plain model gives the same trade-off for free. (c) After about 0.85 the curve falls off a cliff, because the last frauds look like honest payments. In Task 2, recall 0.91 already costs a precision of 0.24, and recall 0.95 a precision of about 0.05: twenty checks for every fraud caught. A small margin above the goal is wise, since recall on new payments varies a little around the cross-validated value; a large one is very expensive. (d) Precision: a false alarm is now expensive, an honest customer is blocked, and nobody checks first. Move the threshold up. The same model serves both purposes, with a different threshold. (e) The solution flags 218 payments, €1,090 of checks, and misses 24 frauds, €2,400: €3,490 in total. "Never fraud" checks nothing and misses all 197 frauds: €19,700. The right threshold depends on what the mistakes cost (Finish early B).

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** One patient in a hundred has a rare disease. A model says "healthy" for everybody. What are its accuracy and its recall?

A. 0.99 and 0.99 &nbsp; B. 0.99 and 0 &nbsp; C. 0.01 and 0 &nbsp; D. 0.5 and 0.5

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** It is right for the 99 healthy patients, so its accuracy is 0.99, but it finds none of the sick ones: recall 0 (Lecture 10, "First, evaluate correctly").

</details>

**2.** A fraud model flags 100 payments; 20 of them are fraud. The data contain 50 frauds. What are precision and recall?

A. 0.20 and 0.40 &nbsp; B. 0.40 and 0.20 &nbsp; C. 0.20 and 0.50 &nbsp; D. 0.80 and 0.40

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Precision: 20 of the 100 flagged are fraud, 0.20. Recall: 20 of the 50 frauds are found, 0.40 (Lecture 9, "Precision and recall").

</details>

**3.** Which of the following is **not** a remedy for class imbalance when you train a model?

A. Sampling the classes in proportion to their share in the data &nbsp; B. Undersampling the majority class &nbsp; C. Oversampling the minority class &nbsp; D. Re-weighting the loss function

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Sampling in proportion to the class shares keeps the imbalance as it is. B, C and D (class weights) are the remedies of Lecture 10.

</details>

**4.** You lower the threshold of a classifier from 0.5 to 0.2. What happens?

A. Recall goes down, precision up &nbsp; B. Recall goes up, precision usually down &nbsp; C. Both go up &nbsp; D. The model has to be trained again

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** More payments are flagged: more of the frauds are found, and more false alarms come in. The model stays the same; only the line moves (Lecture 9, "The threshold is a choice").

</details>

**5.** Which of the following statements about imbalanced data are true?

(a) Undersampling the majority class makes training faster.<br>
(b) Oversampling the majority class helps reduce the complexity of the model.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) **True:** fewer rows to train on. (b) **False:** you oversample the *minority* class, and that does not make the model simpler (Lecture 10).

</details>

**6.** A colleague applies SMOTE to all 30,000 payments first, and then cross-validates a model on the result. Their recall is

A. honest &nbsp; B. too optimistic &nbsp; C. too pessimistic

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The synthetic frauds are made from neighbors, some of which end up in the validation folds: the model is tested on near-copies of what it trained on. Resample the training folds only, with the pipeline of imbalanced-learn (Lecture 10, "Resample the training data only").

</details>

**7.** A data scientist builds a classification model on a large, imbalanced data set. The deadline is close, so her main concern is training time. Which remedy for class imbalance should she avoid?

A. Random undersampling &nbsp; B. Random oversampling &nbsp; C. Class weights &nbsp; D. A balanced random forest

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Oversampling makes the training data even larger: training is slower, and the copies add no information (Lecture 10, "Random undersampling and oversampling").

</details>

**8.** Which of the following is **not** a metric for regression?

A. Accuracy &nbsp; B. Coefficient of determination ($R^2$) &nbsp; C. Root mean squared error (RMSE) &nbsp; D. Mean absolute error (MAE)

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Accuracy counts correct class labels; it is a metric for classification (Lecture 9, "Metrics for regression").

</details>

**9.** You want to compare two models on a rare class before you choose a threshold. Which number do you use?

A. Accuracy &nbsp; B. Recall at 0.5 &nbsp; C. Average precision &nbsp; D. Precision at 0.5

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** Average precision summarizes the precision-recall curve over all thresholds, so it does not depend on one choice of threshold, and it is not fooled by the many easy negatives (Lecture 9, "Comparing models without choosing a threshold").

</details>

**10.** Below are the true labels `t` and the labels predicted by three classification models `c1`, `c2` and `c3`. Compute the accuracy, and the precision, recall and F1-score for class 1, for each model. You design a theft detection system for a supermarket, and you want to avoid false alarms as much as possible, so that customers are not disturbed for nothing. With 1 meaning theft, which model do you choose, and why?

```
t  = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
c1 = [0, 0, 0, 1, 0, 0, 0, 1, 1, 0]
c2 = [1, 1, 0, 0, 1, 0, 1, 0, 1, 1]
c3 = [1, 1, 0, 1, 1, 1, 0, 1, 1, 1]
```

<details><summary><b>Answer</b> (click to unfold)</summary>

| model | TP | FP | FN | TN | accuracy | precision | recall | F1 |
|---|---|---|---|---|---|---|---|---|
| c1 | 2 | 1 | 3 | 4 | 0.60 | 0.67 | 0.40 | 0.50 |
| c2 | 3 | 3 | 2 | 2 | 0.50 | 0.50 | 0.60 | 0.55 |
| c3 | 4 | 4 | 1 | 1 | 0.50 | 0.50 | 0.80 | 0.62 |

A false alarm is the costly mistake here, so precision matters most: choose **c1**. When it flags someone, it is right two times out of three. It misses more thefts, but that is the price of disturbing fewer customers (Lecture 9, "Which metric?").

</details>

## If you finish early

**A. The leak, by hand.** Oversample all 30,000 payments with SMOTE first (`SMOTE(random_state=0).fit_resample(X, y)`), then cross-validate a random forest on the result and look at its recall. Then do it the right way, with SMOTE inside `make_imb_pipeline`. Compare (Lecture 10, "Resample the training data only"). This takes about two minutes.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
X_res, y_res = SMOTE(random_state=0).fit_resample(X, y)                     # SMOTE on everything
print("SMOTE first:           recall", round(cross_val_score(forest, X_res, y_res, cv=5, scoring="recall").mean(), 2))
inside = make_imb_pipeline(SMOTE(random_state=0), forest)
print("SMOTE in the pipeline: recall", round(cross_val_score(inside, X, y, cv=5, scoring="recall").mean(), 2))
# SMOTE first: a recall of 1.0, because the validation folds contain synthetic frauds made from
# the frauds the forest trained on. In the pipeline: about 0.83, the honest answer.

**B. A threshold from costs.** Suppose a check costs the bank €5 of work, and a missed fraud €100. For the logistic regression of Step 1, compute the total cost of every threshold on the honest probabilities `proba`, and find the cheapest. How does its recall compare with the goal of the bank?

In [ ]:
costs = []
for t in [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    flag = proba >= t
    cost = 5 * flag.sum() + 100 * ((~flag) & (y == 1)).sum()
    costs.append((cost, t, recall_score(y, flag), precision_score(y, flag)))
    print(f"threshold {t:5}: cost €{cost:6,}, recall {costs[-1][2]:.2f}, precision {costs[-1][3]:.2f}")
cost, t, rec, prec = min(costs)
print(f"cheapest threshold {t}: cost €{cost:,}")
# The cheapest threshold is about 0.05, with a recall of 0.85: when a missed fraud costs as much as
# twenty checks, the goal of the bank is about what the costs justify. Lower thresholds pay for
# too many checks, higher ones for too many missed frauds.

## Where this is going

> Choose the metric from what the mistakes cost, compare models with a curve, and choose the threshold last.

Today a rare class made accuracy useless. The threshold turned out to be the strongest lever: class weights and resampling moved the point along the curve, and a better model, not a remedy, raised the part of the curve that the bank uses.

**Coming up.** Week 6 (November 30 to December 3): clustering. For the first time there is no label `y` at all.

**Reading.** Müller and Guido (2016), Chapter 5: "Evaluation Metrics and Scoring". Resampling: imbalanced-learn User Guide, "Over-sampling" and "Under-sampling".

---
*This lab practices Lectures 9 and 10 of Ş. İlker Birbil. Data: "Credit Card Fraud Detection" by the Machine Learning Group of the Université Libre de Bruxelles (Dal Pozzolo, Caelen, Johnson and Bontempi, 2015), from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), licensed under the [Open Database License](https://opendatacommons.org/licenses/odbl/1-0/). We use all 492 frauds and a random sample of 49,508 honest payments.*